# 5th attempt - RNN

In [17]:
# 1️⃣ ייבוא המודול והפונקציות
import functions         # ייבוא רגיל למודול
from functions import *  # ייבוא כל הפונקציות ל־namespace

# 2️⃣ טעינה מחדש
import importlib
importlib.reload(functions)  # מוודא שהמודול מעודכן

# 3️⃣ הרצת הפונקציות שוב ל־namespace
from functions import *  # שוב מייבא את כל הפונקציות אחרי ה־reload

In [18]:
import numpy as np
import pandas as pd
from functions import *
from read_from_file_df import *
import pickle
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [ ]:
SIZE = 10
AMOUNT_BOARDS = 1000
AMOUNT_MOVES = 100
NUM_DICT = 1
gen = 2

In [20]:
reverse_df_sort = load_reverse_df(SIZE, AMOUNT_BOARDS, gen)
X_train, X_val, X_test, y_train, y_val, y_test = prepare_reverse_dataset(reverse_df_sort, SIZE, gen, target_pixel_index=0, test_size=0.1, val_size=0.1, random_state=365)
X_train_array, X_val_array, X_test_array, y_train_array, y_val_array, y_test_array = to_numpy_4d(X_train, X_val, X_test, y_train, y_val, y_test, SIZE)

print("len df:", len(reverse_df_sort))
print("len train: ", len(X_train))
print("len val: ",len(X_val))
print("len test: ",len(X_test))

len df: 29760
len train:  24105
len val:  2679
len test:  2976


In [21]:
from functions import build_and_train_rnn

# build and train RNN model
model, history = build_and_train_rnn(
    X_train_array,
    y_train_array,
    size=SIZE,
    gen=gen,
    rnn_units=128,
    dense_units=64,
    epochs=20,
    batch_size=32,
    validation_split=0.2
)

model.summary()


Epoch 1/20
603/603 ━━━━━━━━━━━━━━━━━━━━ 22s 15ms/step - accuracy: 0.6572 - loss: 0.6274 - val_accuracy: 0.6741 - val_loss: 0.6040
Epoch 2/20
603/603 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.6813 - loss: 0.6033 - val_accuracy: 0.6878 - val_loss: 0.5981
Epoch 3/20
603/603 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 0.6943 - loss: 0.5956 - val_accuracy: 0.6918 - val_loss: 0.5943
Epoch 4/20
603/603 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.6877 - loss: 0.5950 - val_accuracy: 0.6992 - val_loss: 0.5905
Epoch 5/20
603/603 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.6967 - loss: 0.5842 - val_accuracy: 0.6918 - val_loss: 0.5907
Epoch 6/20
603/603 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7034 - loss: 0.5782 - val_accuracy: 0.6947 - val_loss: 0.5890
Epoch 7/20
603/603 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7020 - loss: 0.5740 - val_accuracy: 0.7007 - val_loss: 0.5830
Epoch 8/20
603/603 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7112 - loss: 0.5678 - val_accuracy:

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 128)            │        78,848 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 261,509 (1021.52 KB)

 Trainable params: 87,169 (340.50 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 174,340 (681.02 KB)

In [ ]:
X_test_rnn = X_test_array.reshape((-1, gen-1, SIZE*SIZE)).astype('float32')
X_test_rnn = y_test_array.reshape((-1, gen-1, SIZE*SIZE)).astype('float32')

test_loss, test_acc = model.evaluate(X_test_rnn, y_test_array.astype('float32'))
print(f"Test accuracy: {test_acc:.3f}")

Epoch 1/20
603/603 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.7472 - loss: 0.5103 - val_accuracy: 0.6986 - val_loss: 0.5907
Epoch 2/20
603/603 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.7495 - loss: 0.5078 - val_accuracy: 0.7023 - val_loss: 0.5925
Epoch 3/20
603/603 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7564 - loss: 0.4997 - val_accuracy: 0.6982 - val_loss: 0.5947
Epoch 4/20
603/603 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.7564 - loss: 0.4946 - val_accuracy: 0.6992 - val_loss: 0.5960
Epoch 5/20
603/603 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.7622 - loss: 0.4898 - val_accuracy: 0.6945 - val_loss: 0.6023
Epoch 6/20
603/603 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.7670 - loss: 0.4804 - val_accuracy: 0.7001 - val_loss: 0.6056
Epoch 7/20
603/603 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.7690 - loss: 0.4811 - val_accuracy: 0.6855 - val_loss: 0.6177
Epoch 8/20
603/603 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.7756 - loss: 0.4690 - val_accuracy: 0.

In [30]:
evaluate_model(model, X_test_rnn, y_test_array)

93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step

===== Evaluation Results =====
┌──────────────┬────────────┬────────────┐
│              │ Pred=Alive │ Pred=Dead  │
├──────────────┼────────────┼────────────┤
│ True=Alive   │        445 │        576 │
│ True=Dead    │        401 │       1554 │
└──────────────┴────────────┴────────────┘

--- Performance Metrics ---
Accuracy    : 0.672
Precision   : 0.526
Recall      : 0.436
F1-score    : 0.477
